# 30D — Source-balance v2 archived-frame pose audit

**CPU → Run all.** The first 30D run was blocked by YouTube bot checks. This repaired run makes no YouTube request. It downloads the checksum-verified 30B and 30C evidence ZIPs already archived in this repository, reconstructs their original 2-FPS tiles, and measures coarse MediaPipe pose visibility for the 18 approved windows. It exports overlays and coarse audit landmarks only; these are **not training landmarks**. Independent human review and training remain locked.


In [ ]:
%pip install -q mediapipe==0.10.35 opencv-python-headless pandas==2.2.3


In [ ]:
from pathlib import Path
import hashlib,json,shutil,urllib.request,zipfile
import cv2,mediapipe as mp,numpy as np,pandas as pd
from google.colab import files
from IPython.display import display

PINNED_COMMIT='30e24b9ae76d25d232272ebb26275dc920d2474d'
RAW=f'https://raw.githubusercontent.com/bariscelikk1/cALorie/{PINNED_COMMIT}/'
INPUTS={
 'ml/data/source_balance_v2/pose_audit_targets.csv':'480319409c2b6e1fd80e786e22b56ce3bab9ab0d35cba0248494a01d8e4885e4',
 'ml/artifacts/source_balance_v2_dense_and_replacements/calorie_source_balance_v2_dense_and_replacements.zip':'948d0ef9314b2292a4d80fdb1283b9bb935d7458e2c0fc3be3a56646c4c5a0e9',
 'ml/artifacts/source_balance_v2_replacement_dense_review/calorie_source_balance_v2_replacement_dense_review.zip':'56e2b123418300bb8f9983f3a89f92b64bfb4046dbc7083b5123b65b4d462faf',
}
WORK=Path('/content/calorie_source_balance_v2_archived_pose_work')
OUTPUT=Path('/content/calorie_source_balance_v2_pose_audit')
if WORK.exists():shutil.rmtree(WORK)
if OUTPUT.exists():shutil.rmtree(OUTPUT)
for folder in (WORK,WORK/'inputs',WORK/'30b',WORK/'30c',OUTPUT,OUTPUT/'coarse_pose_landmarks',OUTPUT/'pose_contact_sheets'):
 folder.mkdir(parents=True,exist_ok=True)

def sha256(path):
 d=hashlib.sha256()
 with Path(path).open('rb') as handle:
  for block in iter(lambda:handle.read(1024*1024),b''):d.update(block)
 return d.hexdigest()

def fetch(relative,digest):
 path=WORK/'inputs'/Path(relative).name
 urllib.request.urlretrieve(RAW+relative,path)
 assert sha256(path)==digest,(relative,sha256(path))
 return path

paths={name:fetch(name,digest) for name,digest in INPUTS.items()}
targets=pd.read_csv(paths['ml/data/source_balance_v2/pose_audit_targets.csv'])
assert len(targets)==18 and targets.window_id.is_unique
assert (~targets.independent_human_review_complete).all() and (~targets.training_allowed).all()
archives={
 '30b':paths['ml/artifacts/source_balance_v2_dense_and_replacements/calorie_source_balance_v2_dense_and_replacements.zip'],
 '30c':paths['ml/artifacts/source_balance_v2_replacement_dense_review/calorie_source_balance_v2_replacement_dense_review.zip'],
}
for key,archive in archives.items():
 with zipfile.ZipFile(archive) as z:z.extractall(WORK/key)
 status=pd.read_csv(WORK/key/'dense_review_status.csv')
 assert status.review_id.is_unique
 locals()[f'status_{key}']=status

def verify_internal(root):
 lines=(root/'SHA256SUMS.txt').read_text().splitlines()
 for line in lines:
  digest,relative=line.split('  ',1)
  assert sha256(root/relative)==digest,relative
 return len(lines)

verified_30b=verify_internal(WORK/'30b');verified_30c=verify_internal(WORK/'30c')
status_by_review={}
for archive_key,status in (('30b',status_30b),('30c',status_30c)):
 for row in status.itertuples(index=False):
  status_by_review[row.review_id]={'archive_key':archive_key,'start_s':float(row.start_s),'end_s':float(row.end_s),'status':row.status}
assert set(targets.source_review_id).issubset(set(status_by_review))
for row in targets.itertuples(index=False):
 source=status_by_review[row.source_review_id]
 assert source['status']=='evidence_ready',row.window_id
 assert source['start_s']<=row.start_s<row.end_s<=source['end_s'],row.window_id
print('Verified archive payloads:',verified_30b,'30B +',verified_30c,'30C')
print('CPU repaired 30D:',len(targets),'windows from archived 2-FPS evidence; no YouTube and no training')
display(targets.groupby(['target_class','video_id']).size().rename('windows').reset_index())


In [ ]:
MODEL=WORK/'pose_landmarker_full.task'
MODEL_URL=('https://storage.googleapis.com/mediapipe-models/pose_landmarker/'
 'pose_landmarker_full/float16/latest/pose_landmarker_full.task')
urllib.request.urlretrieve(MODEL_URL,MODEL)
assert MODEL.stat().st_size>1_000_000
EVIDENCE_FPS=2.0
KEEP_17=[0,2,5,7,8,11,12,13,14,15,16,23,24,25,26,27,28]
CRITICAL=[11,12,23,24,25,26,27,28]
SKELETON=[(11,13),(13,15),(12,14),(14,16),(11,12),(11,23),(12,24),(23,24),(23,25),(25,27),(24,26),(26,28)]

def normalize_pose(frame17):
 xyz=frame17[:,:3].copy();hip=(xyz[11]+xyz[12])/2;shoulder=(xyz[5]+xyz[6])/2
 scale=max(float(np.linalg.norm(xyz[5]-xyz[6])),float(np.linalg.norm(shoulder-hip)),1e-6)
 return np.concatenate([(xyz-hip)/scale,frame17[:,3:4]],axis=1)

def iter_tiles(sheet_root,locator_start,start_s,end_s):
 index=0
 for page_path in sorted(sheet_root.glob('*.jpg')):
  page=cv2.imread(str(page_path));assert page is not None,page_path
  height,width=page.shape[:2];tile_h=height//3;tile_w=width//4
  assert tile_h>=180 and tile_w>=320,(page_path,page.shape)
  for row in range(3):
   for col in range(4):
    timestamp=locator_start+index/EVIDENCE_FPS
    tile=page[row*tile_h:(row+1)*tile_h,col*tile_w:(col+1)*tile_w].copy()
    if start_s<=timestamp<end_s:yield timestamp,tile
    index+=1

def draw_overlay(frame,points,timestamp):
 tile=cv2.resize(frame,(320,180))
 if points:
  for a,b in SKELETON:
   if min(points[a].visibility,points[b].visibility)>=.5:
    cv2.line(tile,(int(points[a].x*320),int(points[a].y*180)),(int(points[b].x*320),int(points[b].y*180)),(0,255,0),2)
  for point in points:
   if point.visibility>=.5:cv2.circle(tile,(int(point.x*320),int(point.y*180)),2,(0,0,255),-1)
 cv2.rectangle(tile,(0,0),(104,25),(0,0,0),-1)
 cv2.putText(tile,f'{int(timestamp//60):02d}:{timestamp%60:04.1f}',(4,18),cv2.FONT_HERSHEY_SIMPLEX,.5,(0,255,255),1)
 return tile

def audit_window(row,landmarker):
 source=status_by_review[row.source_review_id];root=WORK/source['archive_key']
 sheet_root=root/'dense_contact_sheets'/row.source_review_id
 assert sheet_root.exists(),sheet_root
 raw=[];normalized=[];detected=[];timestamps=[];overlays=[]
 for timestamp,frame in iter_tiles(sheet_root,source['start_s'],float(row.start_s),float(row.end_s)):
  rgb=cv2.cvtColor(frame,cv2.COLOR_BGR2RGB)
  result=landmarker.detect(mp.Image(image_format=mp.ImageFormat.SRGB,data=rgb))
  points=result.pose_landmarks[0] if result.pose_landmarks else None
  if points:
   full=np.asarray([[p.x,p.y,p.z,float(p.visibility or 0)] for p in points],np.float32)
   raw.append(full);normalized.append(normalize_pose(full[KEEP_17]));detected.append(True)
  else:
   raw.append(np.full((33,4),np.nan,np.float32));normalized.append(np.full((17,4),np.nan,np.float32));detected.append(False)
  timestamps.append(timestamp);overlays.append(draw_overlay(frame,points,timestamp))
 assert timestamps,row.window_id
 raw=np.stack(raw);normalized=np.stack(normalized);detected=np.asarray(detected,bool);visibility=raw[:,:,3]
 np.savez_compressed(OUTPUT/'coarse_pose_landmarks'/(row.window_id+'.npz'),
  image_landmarks=raw,landmarks=normalized,timestamps_s=np.asarray(timestamps,np.float32),detected=detected,
  evidence_fps=np.float32(EVIDENCE_FPS),video_id=row.video_id,target_class=row.target_class,
  training_allowed=False,not_training_landmarks=True,evidence_source='archived_contact_sheet_tiles')
 page_root=OUTPUT/'pose_contact_sheets'/row.window_id;page_root.mkdir(parents=True,exist_ok=True)
 for page_index,offset in enumerate(range(0,len(overlays),12)):
  tiles=overlays[offset:offset+12]
  while len(tiles)<12:tiles.append(np.zeros_like(tiles[0]))
  page=np.vstack([np.hstack(tiles[i:i+4]) for i in (0,4,8)])
  cv2.imwrite(str(page_root/f'{page_index:03d}.jpg'),page,[cv2.IMWRITE_JPEG_QUALITY,86])
 joint_rates=np.nanmean(visibility>=.5,axis=0) if detected.any() else np.zeros(33)
 full_body=np.all(visibility[:,CRITICAL]>=.5,axis=1) if detected.any() else np.zeros(len(raw),bool)
 return {'frames':len(timestamps),'detection_rate':float(detected.mean()),
  'visible_joint_rate_33':float(np.nanmean(visibility>=.5)) if detected.any() else 0.0,
  'full_body_frame_rate':float(full_body.mean()),'left_ankle_visible_rate':float(joint_rates[27]),
  'right_ankle_visible_rate':float(joint_rates[28]),'pose_pages':len(list(page_root.glob('*.jpg'))),
  'landmark_path':f'coarse_pose_landmarks/{row.window_id}.npz'}


In [ ]:
options=mp.tasks.vision.PoseLandmarkerOptions(
 base_options=mp.tasks.BaseOptions(model_asset_path=str(MODEL)),
 running_mode=mp.tasks.vision.RunningMode.IMAGE,num_poses=1,
 min_pose_detection_confidence=.5,min_pose_presence_confidence=.5,
 output_segmentation_masks=False)
rows=[]
with mp.tasks.vision.PoseLandmarker.create_from_options(options) as landmarker:
 for row in targets.itertuples(index=False):
  record=row._asdict();record.update(status='pose_failed',error='')
  try:record.update(audit_window(row,landmarker),status='pose_ready')
  except Exception as exc:record['error']=f'{type(exc).__name__}: {exc}'[:500]
  rows.append(record);print(row.window_id,record['status'],record.get('frames',0),record.get('detection_rate',0))
quality=pd.DataFrame(rows)
quality['review_status']='draft';quality['independent_human_review_complete']=False;quality['training_allowed']=False
quality.to_csv(OUTPUT/'pose_quality.csv',index=False)
targets.to_csv(OUTPUT/'candidate_windows.csv',index=False)
archive_status=pd.DataFrame([
 {'archive':'30b','sha256':sha256(archives['30b']),'verified_payload_files':verified_30b},
 {'archive':'30c','sha256':sha256(archives['30c']),'verified_payload_files':verified_30c},
])
archive_status.to_csv(OUTPUT/'archive_status.csv',index=False)
metadata={'notebook':'30d_colab_source_balance_v2_pose_audit.ipynb','mode':'archived_2fps_fallback',
 'runtime':'CPU','pinned_commit':PINNED_COMMIT,'evidence_fps':EVIDENCE_FPS,'window_count':len(targets),
 'pose_ready':int((quality.status=='pose_ready').sum()),'youtube_requests':0,'raw_video_exported':False,
 'training_landmarks_exported':False,'independent_human_review_complete':False,'training_allowed':False,
 'next_step':'review coarse pose overlays and metrics before any 15-FPS extraction or training'}
(OUTPUT/'run_metadata.json').write_text(json.dumps(metadata,indent=2)+'\n')
checks=[]
for path in sorted(p for p in OUTPUT.rglob('*') if p.is_file() and p.name!='SHA256SUMS.txt'):
 checks.append(f'{sha256(path)}  {path.relative_to(OUTPUT)}')
(OUTPUT/'SHA256SUMS.txt').write_text('\n'.join(checks)+'\n')
archive=shutil.make_archive(str(OUTPUT),'zip',OUTPUT)
display(quality[['window_id','status','frames','detection_rate','visible_joint_rate_33','full_body_frame_rate','left_ankle_visible_rate','right_ankle_visible_rate']])
print('Created:',archive,round(Path(archive).stat().st_size/1e6,1),'MB');files.download(archive)


This is a **coarse 2-FPS visibility audit** derived from archived review frames. It cannot replace continuous 15-FPS motion landmarks and must never be added to training. Use it only to reject weak pose coverage and prioritize windows for later source-backed extraction and independent human review.
